# Laboratori 7 — Rrugët më të Shkurtra: Dijkstra · Bellman-Ford · Floyd-Warshall

---

## Algoritmet që trajtohen

Në këtë laborator do të shqyrtohen dhe implementohen tre algoritmet kryesore për gjetjen e rrugëve më të shkurtra në grafe me peshe.

| Algoritmi | Peshe negative | Burim | Kompleksiteti |
|-----------|:--------------:|:-----:|:-------------:|
| **Dijkstra** | (vetëm ≥ 0) | i vetëm | O((V+E) log V) |
| **Bellman-Ford** | (zbulim cikli) | i vetëm | O(V·E) |
| **Floyd-Warshall** | | të gjitha | O(V³) |

---

### Dijkstra

**Ideja kryesore:** zgjedhim gjithmonë kulmin me distancën me te vogel nga kulmet qe nuk jane tabeluar(kulmet qe nuk kane marre tabelimin e perhershem)

**Kusht kritik:** të gjitha peshet duhet të jenë **jo-negative** (w ≥ 0).

**Hapat:**
1. dist[src] = 0, të gjitha të tjerat = ∞.
2. Futim src në tabelim.
3. kulmin me distance minimale min(dist) qe po e shenojme me "u" → shënojmë si "vizituar".
4. Për çdo fqinj "v" te kulmit "u": nëse dist[v]>dist[u]+w(u,v)  → perditesojme vleren e distances se kulmit "v" dhe paraardhesin e tij.
5. Vazhdojmë deri sa te gjithe kulmet e grafit te tabelohen.

```
Kompleksiteti: O((V + E) · log V)  
```

---

### Bellman-Ford

**Ideja kryesore:** peerditesojme paraardhesin dhe distancen e cdo kulmi duke perdorur **të gjitha brinjët** V-1 herë. Kjo siguron që distancat konvergjojnë edhe me peshe negative.

**Detektimi i cikleve negative:** nëse pas V-1 iterimeve mund të peerditesojee akoma distance dhe paraardhes atehere ka cikël negativ.

```
Kompleksiteti: O(V · E)
```

---

### Floyd-Warshall

**Ideja kryesore:** programim dinamik. Për çdo kulm ndërmjetës k, provojmë nëse rruga i→k→j është më e shkurtër se i→j.

**Gjen distancat mes TË GJITHA çifteve** të kulmeve njëherësh.

```
D[i][j] = min(D[i][j],  D[i][k] + D[k][j])
Kompleksiteti: O(V³)
```

---

### Krahasimi

| Karakteristika | Dijkstra | Bellman-Ford | Floyd-Warshall |
|----------------|----------|-------------|---------------|
| Peshe negative | ❌ | ✅ | ✅ |
| Cikël negativ | ❌ zbulim | ✅ zbulim | ✅ zbulim |
| Burim(e) | i vetëm | i vetëm | të gjitha çiftet |
| Kompleksiteti | O((V+E)logV) | O(VE) | O(V³) |
| Zbatim real | GPS, rrugë | Arbitrazh, rrjete | Routing tabelë |

---

---

## Ushtrimi 1 — Dijkstra GPS: Harta e Shqipërisë 🗺

### Teoria: Dijkstra në navigimin GPS

Algoritmi **Dijkstra** përdoret gjerësisht në sistemet GPS (si Google Maps, Waze) për gjetjen e rrugës më të shkurtër midis dy qyteteve. Çdo rrugë ka një **peshe** (distancë ose kohë udhëtimi) dhe Dijkstra garanton gjetjen e rrugës optimale **kur peshet janë jo-negative**.

### Animacioni

Kjo simulim paraqet **hartën e Shqipërisë** me 12 qytete kryesore dhe rrugët lidhëse (me distanca reale km). Algoritmi Dijkstra ekzekutohet **hap-pas-hapi** ndërsa një **makinë animohet** gjatë rrugës optimale.

**Kontrollet:**
- Klik qytet burim → klik destinacion
- `SPACE` — hap tjetër i Dijkstrës
- `A` — auto-luaj (animim automatik)
- `ENTER` — nis makinën në rrugën optimale
- `R` — rifillo | `ESC` — dil

In [1]:
import pygame, heapq, math, sys, time

pygame.init()
W, H = 1280, 720
screen = pygame.display.set_mode((0, 0), pygame.FULLSCREEN)
W, H = screen.get_size()
pygame.display.set_caption("Ushtrimi 1 — Dijkstra GPS: Harta e Shqipërisë")
clock = pygame.time.Clock()

# ── colours ──────────────────────────────────────────────────────────────────
BG        = (10,  16,  30)
ROAD_COL  = (50,  60,  80)
ROAD_ACT  = (80, 130, 200)
PATH_COL  = (255, 220,  50)
NODE_COL  = (40,  80, 140)
NODE_SEL  = (255, 160,  30)
NODE_VIS  = (60, 180,  80)
NODE_DONE = (200, 220, 255)
TEXT_COL  = (220, 230, 255)
PANEL_BG  = (15,  22,  42)
CAR_COL   = (255,  80,  80)
WHITE     = (255, 255, 255)
GOLD      = (255, 215,   0)
GREEN     = (80, 200, 100)
RED       = (220,  60,  60)

# ── fonts ─────────────────────────────────────────────────────────────────────
F_BIG   = pygame.font.SysFont("consolas", 22, bold=True)
F_MED   = pygame.font.SysFont("consolas", 16)
F_SML   = pygame.font.SysFont("consolas", 13)
F_TITLE = pygame.font.SysFont("consolas", 28, bold=True)

# ── graph: Albanian cities (pixel positions on a 900×720 canvas) ──────────────
CITIES = {
    "Shkodër":   (210, 80),
    "Lezhë":     (240, 160),
    "Tiranë":    (270, 270),
    "Durrës":    (180, 290),
    "Elbasan":   (370, 320),
    "Berat":     (320, 430),
    "Lushnjë":   (250, 400),
    "Fier":      (230, 490),
    "Vlorë":     (180, 580),
    "Gjirokastër":(340, 600),
    "Korçë":     (490, 450),
    "Pogradec":  (450, 360),
}

EDGES = [
    ("Shkodër",    "Lezhë",       45),
    ("Lezhë",      "Tiranë",      90),
    ("Lezhë",      "Durrës",      70),
    ("Tiranë",     "Durrës",      38),
    ("Tiranë",     "Elbasan",     54),
    ("Durrës",     "Lushnjë",     70),
    ("Elbasan",    "Pogradec",    70),
    ("Elbasan",    "Berat",       80),
    ("Lushnjë",    "Berat",       45),
    ("Lushnjë",    "Fier",        30),
    ("Fier",       "Vlorë",       50),
    ("Fier",       "Berat",       40),
    ("Berat",      "Gjirokastër", 110),
    ("Vlorë",      "Gjirokastër", 100),
    ("Gjirokastër","Korçë",       130),
    ("Korçë",      "Pogradec",    50),
]

# build adjacency
graph = {c: [] for c in CITIES}
for u, v, w in EDGES:
    graph[u].append((v, w))
    graph[v].append((u, w))

city_names = list(CITIES.keys())

# ── Dijkstra (returns steps for animation) ───────────────────────────────────
def dijkstra_steps(start, end):
    dist   = {c: float('inf') for c in CITIES}
    prev   = {c: None for c in CITIES}
    dist[start] = 0
    pq     = [(0, start)]
    visited = set()
    steps  = []   # each step: (dist copy, visited copy, pq copy, relaxed_edge)
    while pq:
        d, u = heapq.heappop(pq)
        if u in visited:
            continue
        visited.add(u)
        steps.append(("visit", u, dict(dist), set(visited), [(x[1],x[0]) for x in pq]))
        if u == end:
            break
        for v, w in graph[u]:
            nd = d + w
            if nd < dist[v]:
                dist[v] = nd
                prev[v] = u
                heapq.heappush(pq, (nd, v))
                steps.append(("relax", u, v, w, dict(dist), set(visited)))
    # reconstruct path
    path = []
    cur = end
    while cur:
        path.append(cur)
        cur = prev[cur]
    path.reverse()
    return steps, path, dist

# ── state ──────────────────────────────────────────────────────────────────────
src, dst = None, None
phase     = "pick_src"   # pick_src → pick_dst → running → animating → done
steps     = []
step_idx  = 0
path      = []
dist_final= {}
visited_anim = set()
dist_anim    = {}
auto_play    = False
last_auto    = 0

# car animation
car_pos   = None
car_seg   = 0
car_t     = 0.0

def lerp(a, b, t):
    return (a[0] + (b[0]-a[0])*t, a[1] + (b[1]-a[1])*t)

def draw_arrow(surface, color, start, end, width=3):
    dx, dy = end[0]-start[0], end[1]-start[1]
    length = math.hypot(dx, dy)
    if length < 1: return
    ux, uy = dx/length, dy/length
    pygame.draw.line(surface, color, (int(start[0]), int(start[1])),
                     (int(end[0]), int(end[1])), width)
    # arrowhead
    ax = end[0] - ux*14 - uy*7
    ay = end[1] - uy*14 + ux*7
    bx = end[0] - ux*14 + uy*7
    by = end[1] - uy*14 - ux*7
    pygame.draw.polygon(surface, color, [(int(end[0]),int(end[1])),
                                          (int(ax),int(ay)),(int(bx),int(by))])

def draw_car(surface, pos, color):
    x, y = int(pos[0]), int(pos[1])
    # body
    pygame.draw.rect(surface, color,       (x-14, y-7,  28, 14), border_radius=5)
    pygame.draw.rect(surface, WHITE,       (x- 7, y-12, 14,  8), border_radius=3)
    # wheels
    for wx in [x-9, x+6]:
        pygame.draw.circle(surface, (30,30,30), (wx, y+7), 5)
        pygame.draw.circle(surface, (180,180,180),(wx, y+7), 3)

def draw_scene(step_desc="", dist_show=None, vis_show=None, pq_show=None):
    # map area 0..900, panel 900..1280
    MAP_W = 900
    screen.fill(BG)
    pygame.draw.rect(screen, PANEL_BG, (MAP_W, 0, W-MAP_W, H))
    pygame.draw.line(screen, (40,60,90), (MAP_W,0),(MAP_W,H), 2)

    # draw edges
    for u, v, w in EDGES:
        p1, p2 = CITIES[u], CITIES[v]
        in_path = path and u in path and v in path and (
            abs(path.index(u)-path.index(v)) == 1)
        col = PATH_COL if in_path else (ROAD_ACT if (vis_show and u in vis_show and v in vis_show) else ROAD_COL)
        lw = 5 if in_path else 3
        pygame.draw.line(screen, col, p1, p2, lw)
        # weight label
        mx, my = (p1[0]+p2[0])//2, (p1[1]+p2[1])//2
        wt = F_SML.render(str(w), True, (140,160,200))
        screen.blit(wt, (mx-10, my-8))

    # draw nodes
    for name, pos in CITIES.items():
        if vis_show and name in vis_show:
            col = NODE_DONE
        elif name == src:
            col = NODE_SEL
        elif name == dst:
            col = GREEN
        else:
            col = NODE_COL
        pygame.draw.circle(screen, col, pos, 18)
        pygame.draw.circle(screen, WHITE, pos, 18, 2)
        lbl = F_SML.render(name, True, WHITE)
        screen.blit(lbl, (pos[0]-lbl.get_width()//2, pos[1]+20))
        # distance label
        if dist_show and name in dist_show and dist_show[name] < float('inf'):
            d_lbl = F_SML.render(str(dist_show[name]), True, GOLD)
            screen.blit(d_lbl, (pos[0]-d_lbl.get_width()//2, pos[1]-30))

    # car
    if car_pos:
        draw_car(screen, car_pos, CAR_COL)

    # ── right panel ───────────────────────────────────────────────────────────
    px = MAP_W + 10
    py = 10

    title = F_BIG.render("🗺  Dijkstra GPS", True, GOLD)
    screen.blit(title, (px, py)); py += 36

    # instructions
    if phase == "pick_src":
        inst = ["Kliko qytetin", "BURIMIT (src)"]
    elif phase == "pick_dst":
        inst = ["Kliko qytetin", "DESTINACIONIT (dst)"]
    elif phase == "running":
        inst = ["SPACE = hap tjetër",  "A = auto-luaj", "ENTER = animim auto", "R = rifillo"]
    else:
        inst = ["ENTER = animim makine", "R = rifillo"]
    for ln in inst:
        s = F_SML.render(ln, True, (150,170,210))
        screen.blit(s, (px, py)); py += 18
    py += 8

    # source / dest
    lbl_s = F_MED.render(f"Burimi : {src or '—'}", True, NODE_SEL if src else (120,130,150))
    lbl_d = F_MED.render(f"Dest   : {dst or '—'}", True, GREEN if dst else (120,130,150))
    screen.blit(lbl_s, (px, py)); py += 22
    screen.blit(lbl_d, (px, py)); py += 30

    pygame.draw.line(screen, (40,60,90), (px, py), (W-10, py)); py += 8

    # distance table
    header = F_MED.render("Distancat:", True, (180,200,240))
    screen.blit(header, (px, py)); py += 22
    if dist_show:
        sorted_cities = sorted(dist_show.items(), key=lambda x: (x[1]==float('inf'), x[1]))
        for city, d in sorted_cities:
            ds = "∞" if d == float('inf') else str(d)
            col = GOLD if city == dst and d < float('inf') else TEXT_COL
            row = F_SML.render(f"  {city[:10]:<11} {ds:>4} km", True, col)
            screen.blit(row, (px, py)); py += 17
    py += 8

    pygame.draw.line(screen, (40,60,90), (px, py), (W-10, py)); py += 8

    # step message
    if step_desc:
        words = step_desc.split()
        line, lines = "", []
        for w in words:
            if len(line)+len(w)+1 > 22:
                lines.append(line); line = w
            else:
                line = (line+" "+w).strip()
        if line: lines.append(line)
        for ln in lines:
            s = F_SML.render(ln, True, (220, 240, 180))
            screen.blit(s, (px, py)); py += 17

    # path result
    if path and phase in ("animating", "done"):
        py += 8
        pygame.draw.line(screen, (40,60,90), (px, py), (W-10, py)); py += 8
        s = F_MED.render("Rruga optimale:", True, GOLD)
        screen.blit(s, (px, py)); py += 22
        for i, c in enumerate(path):
            arrow = "→ " if i > 0 else "✦ "
            s2 = F_SML.render(arrow+c, True, PATH_COL)
            screen.blit(s2, (px, py)); py += 17
        if dist_final and dst in dist_final:
            s3 = F_MED.render(f"Total: {dist_final[dst]} km", True, GREEN)
            screen.blit(s3, (px, py))

    # title at top of map
    title_map = F_TITLE.render("Harta e Shqipërisë", True, WHITE)
    screen.blit(title_map, (MAP_W//2 - title_map.get_width()//2, 10))

    # ESC hint
    esc = F_SML.render("ESC = dil", True, (80,90,110))
    screen.blit(esc, (MAP_W+10, H-20))

    pygame.display.flip()

def reset():
    global src, dst, phase, steps, step_idx, path, dist_final
    global visited_anim, dist_anim, auto_play, car_pos, car_seg, car_t
    src=dst=None; phase="pick_src"; steps=[]; step_idx=0
    path=[]; dist_final={}; visited_anim=set(); dist_anim={}
    auto_play=False; car_pos=None; car_seg=0; car_t=0.0

reset()
step_msg = ""

running = True
while running:
    dt = clock.tick(60) / 1000.0

    for event in pygame.event.get():
        if event.type == pygame.QUIT or (
                event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE):
            running = False

        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_r:
                reset(); step_msg = ""

            elif event.key == pygame.K_SPACE and phase == "running":
                # advance one step
                if step_idx < len(steps):
                    s = steps[step_idx]
                    if s[0] == "visit":
                        _, u, d, vis, pq = s
                        dist_anim = d; visited_anim = vis
                        step_msg = f"Vizitoj: {u} (dist={d[u]} km)"
                    else:
                        _, u, v, w, d, vis = s
                        dist_anim = d; visited_anim = vis
                        step_msg = f"Relaksoj: {u}→{v} (+{w}) = {d[v]} km"
                    step_idx += 1
                else:
                    phase = "done"
                    step_msg = "Dijkstra përfundoi! Shtyp ENTER për animimin."

            elif event.key == pygame.K_a and phase == "running":
                auto_play = not auto_play
                step_msg = "Auto-luaj: " + ("ON" if auto_play else "OFF")

            elif event.key == pygame.K_RETURN and phase in ("running","done"):
                if path:
                    phase = "animating"
                    car_seg = 0; car_t = 0.0
                    car_pos = CITIES[path[0]]

        elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            mx, my = event.pos
            if mx < 900:
                for name, pos in CITIES.items():
                    if math.hypot(mx-pos[0], my-pos[1]) < 22:
                        if phase == "pick_src":
                            src = name; phase = "pick_dst"
                            step_msg = f"Burimi: {name}. Tani kliko destinacionin."
                        elif phase == "pick_dst" and name != src:
                            dst = name; phase = "running"
                            steps, path, dist_final = dijkstra_steps(src, dst)
                            dist_anim = {c: float('inf') for c in CITIES}
                            dist_anim[src] = 0
                            step_msg = f"Dijkstra nga {src} → {dst}. Shtyp SPACE."
                        break

    # auto-play
    if phase == "running" and auto_play:
        now = time.time()
        if now - last_auto > 0.35:
            last_auto = now
            if step_idx < len(steps):
                s = steps[step_idx]
                if s[0] == "visit":
                    _, u, d, vis, pq = s
                    dist_anim = d; visited_anim = vis
                    step_msg = f"Vizitoj: {u}  dist={d[u]} km"
                else:
                    _, u, v, w, d, vis = s
                    dist_anim = d; visited_anim = vis
                    step_msg = f"Relaksoj {u}→{v} = {d[v]} km"
                step_idx += 1
            else:
                auto_play = False
                phase = "done"
                step_msg = "Gati! Shtyp ENTER për animimin e makinës."

    # car animation
    if phase == "animating" and path and car_seg < len(path)-1:
        SPEED = 0.7  # segments per second
        car_t += dt * SPEED
        if car_t >= 1.0:
            car_t -= 1.0
            car_seg += 1
            if car_seg >= len(path)-1:
                phase = "done"
                car_pos = CITIES[path[-1]]
                step_msg = f"Arritur! {src} → {dst} = {dist_final.get(dst,'?')} km"
        if car_seg < len(path)-1:
            p1 = CITIES[path[car_seg]]
            p2 = CITIES[path[car_seg+1]]
            car_pos = lerp(p1, p2, car_t)

    draw_scene(step_msg, dist_anim if dist_anim else None,
               visited_anim if visited_anim else None)

pygame.quit()
sys.exit()


pygame 2.6.1 (SDL 2.28.4, Python 3.13.13)
Hello from the pygame community. https://www.pygame.org/contribute.html


SystemExit: 

c:\Users\Admin\Desktop\Struktura Diskrete dhe Algoritme\Anglisht\Laboratory\pygame_env\Lib\site-packages\IPython\core\interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


---

## Ushtrimi 2 — Dijkstra Interaktiv: Ndërto Grafin Tënd 

### Teoria: Dijkstra dhe peshet jo-negative

Ky ushtrim thekson **kufizimin kryesor** të Dijkstrës: peshet duhet të jenë **≥ 0**. Nëse futim peshë negative, algoritmi mund të japë rezultate **të gabuara** sepse mund të "humbasë" rrugë të shkurtëra.

**Pse Dijkstra dështon me peshe negative?**
Kur vizitojmë një kulm, e shënojmë si "final" — nuk e rishqyrtojmë. Por peshë negative mund të ulë distancën e tij **pas** vizitës.

### Animacioni

Ndërtoni grafin tuaj duke shtuar nyje dhe brinjë me peshe sipas dëshirës. Provoni të futni peshe **negative** dhe shikoni paralajmërimin!

**Kontrollet:**
- `[1]` Shto nyje (klik)  
- `[2]` Shto brinjë (klik dy nyje, fut peshën)
- `[3]` Cakto burimin  `[4]` Cakto destinacionin
- `[S]` Ekzekuto Dijkstrën  `[N]` Hap-pas-hapi
- Klik i djathtë → fshi nyjen | `R` → rifillo | `ESC` → dil

In [3]:
import pygame, heapq, math, sys

pygame.init()
W, H = 1280, 720
screen = pygame.display.set_mode((0, 0), pygame.FULLSCREEN)
W, H = screen.get_size()
pygame.display.set_caption("Ushtrimi 2 — Dijkstra Interaktiv")
clock = pygame.time.Clock()

BG       = (10, 16, 30)
PANEL_BG = (15, 22, 42)
NODE_C   = (50, 110, 200)
NODE_SEL = (255, 160,  30)
NODE_VIS = (60, 200,  80)
NODE_SRC = (255, 80,  80)
NODE_DST = (80, 220, 120)
EDGE_C   = (60, 80, 120)
EDGE_ACT = (100, 160, 255)
PATH_C   = (255, 220,  50)
NEG_C    = (255,  60,  60)
WHITE    = (255, 255, 255)
GOLD     = (255, 215,   0)
GREEN    = (80, 200, 100)
TEXT_C   = (210, 225, 255)

F_TITLE = pygame.font.SysFont("consolas", 24, bold=True)
F_BIG   = pygame.font.SysFont("consolas", 18, bold=True)
F_MED   = pygame.font.SysFont("consolas", 15)
F_SML   = pygame.font.SysFont("consolas", 13)

# ── state ─────────────────────────────────────────────────────────────────────
nodes      = {}    # id -> (x, y)
edges      = []    # (u, v, weight)
node_id    = 0
mode       = "add_node"  # add_node | add_edge | set_src | set_dst | result
sel_node   = None
src_node   = None
dst_node   = None
neg_warning = False
path       = []
dist_result= {}
vis_result = set()
step_steps = []
step_idx   = 0
step_dist  = {}
step_vis   = set()
step_msg   = ""
input_mode = False
input_buf  = ""
pending_edge = None   # (u, v) waiting for weight input
error_msg  = ""
error_timer = 0

GRAPH_W = 880

def get_adj():
    adj = {n: [] for n in nodes}
    for u, v, w in edges:
        adj[u].append((v, w))
        adj[v].append((u, w))
    return adj

def dijkstra(start, end, adj):
    dist = {n: float('inf') for n in nodes}
    prev = {n: None for n in nodes}
    dist[start] = 0
    pq = [(0, start)]
    visited = set()
    steps = []
    while pq:
        d, u = heapq.heappop(pq)
        if u in visited: continue
        visited.add(u)
        steps.append(("visit", u, dict(dist), set(visited)))
        if u == end: break
        for v, w in adj.get(u, []):
            nd = d + w
            if nd < dist[v]:
                dist[v] = nd; prev[v] = u
                heapq.heappush(pq, (nd, v))
                steps.append(("relax", u, v, dict(dist), set(visited)))
    path_ = []
    cur = end
    while cur is not None:
        path_.append(cur); cur = prev[cur]
    path_.reverse()
    if path_ and path_[0] != start: path_ = []
    return steps, path_, dist

def has_negative():
    return any(w < 0 for _, _, w in edges)

def draw_edge(u, v, w, color, width=2):
    p1, p2 = nodes[u], nodes[v]
    pygame.draw.line(screen, color, p1, p2, width)
    mx, my = (p1[0]+p2[0])//2, (p1[1]+p2[1])//2
    neg = w < 0
    wlbl = F_SML.render(str(w), True, NEG_C if neg else (180,200,240))
    screen.blit(wlbl, (mx+4, my-10))

def draw_scene():
    screen.fill(BG)
    pygame.draw.rect(screen, PANEL_BG, (GRAPH_W, 0, W-GRAPH_W, H))
    pygame.draw.line(screen, (40,60,90),(GRAPH_W,0),(GRAPH_W,H),2)

    # edges
    path_set = set()
    if path:
        for i in range(len(path)-1):
            path_set.add((path[i], path[i+1]))
            path_set.add((path[i+1], path[i]))

    for u, v, w in edges:
        in_path = (u,v) in path_set
        col = PATH_C if in_path else (NEG_C if w<0 else EDGE_C)
        lw  = 4 if in_path else 2
        draw_edge(u, v, w, col, lw)

    # pending edge while drawing
    if mode == "add_edge" and sel_node is not None:
        mx, my = pygame.mouse.get_pos()
        if mx < GRAPH_W:
            pygame.draw.line(screen, (80,120,180), nodes[sel_node], (mx,my), 2)

    # nodes
    for nid, pos in nodes.items():
        if nid == src_node: col = NODE_SRC
        elif nid == dst_node: col = NODE_DST
        elif nid in step_vis: col = NODE_VIS
        elif nid == sel_node: col = NODE_SEL
        else: col = NODE_C
        pygame.draw.circle(screen, col, pos, 20)
        pygame.draw.circle(screen, WHITE, pos, 20, 2)
        lbl = F_MED.render(str(nid), True, WHITE)
        screen.blit(lbl, (pos[0]-lbl.get_width()//2, pos[1]-lbl.get_height()//2))
        # dist label
        if step_dist and nid in step_dist and step_dist[nid] < float('inf'):
            dl = F_SML.render(str(step_dist[nid]), True, GOLD)
            screen.blit(dl, (pos[0]-dl.get_width()//2, pos[1]+22))

    # neg warning banner
    if neg_warning:
        ban = pygame.Surface((GRAPH_W, 48), pygame.SRCALPHA)
        ban.fill((200, 30, 30, 180))
        screen.blit(ban, (0, H-50))
        wt = F_BIG.render("⚠  KUJDES: Dijkstra kërkon peshe ≥ 0 !  Peshë negative u zbulua!", True, WHITE)
        screen.blit(wt, (GRAPH_W//2 - wt.get_width()//2, H-42))

    # input box
    if input_mode:
        pygame.draw.rect(screen, (20,30,55), (GRAPH_W//2-150, H//2-40, 300, 80), border_radius=10)
        pygame.draw.rect(screen, GOLD,       (GRAPH_W//2-150, H//2-40, 300, 80), 2, border_radius=10)
        pt = F_MED.render(f"Pesho e brinjes {pending_edge}: {input_buf}_", True, WHITE)
        screen.blit(pt, (GRAPH_W//2 - pt.get_width()//2, H//2-15))
        hint = F_SML.render("ENTER = konfirmo  |  Negative OK (teste Dijkstra)", True, (160,180,220))
        screen.blit(hint, (GRAPH_W//2 - hint.get_width()//2, H//2+20))

    # error
    if error_msg:
        em = F_MED.render(error_msg, True, NEG_C)
        screen.blit(em, (10, H-25))

    # right panel
    px, py = GRAPH_W+10, 10
    title = F_TITLE.render("Dijkstra Interaktiv", True, GOLD)
    screen.blit(title, (px, py)); py += 36

    modes = [
        ("1", "Shto nyje (klik)", mode=="add_node"),
        ("2", "Shto brinj",       mode=="add_edge"),
        ("3", "Cakto burimin",    mode=="set_src"),
        ("4", "Cakto dest.",      mode=="set_dst"),
        ("S", "Ekzekuto!",        False),
        ("N", "Hapi tjetër",      False),
        ("R", "Rifillo",          False),
    ]
    for key, label, active in modes:
        col = GOLD if active else TEXT_C
        s = F_SML.render(f"[{key}] {label}", True, col)
        screen.blit(s, (px, py)); py += 20
    py += 10

    pygame.draw.line(screen, (40,60,90),(px,py),(W-10,py)); py+=8

    s = F_MED.render(f"Nyjet : {len(nodes)}", True, TEXT_C)
    screen.blit(s, (px, py)); py += 20
    s = F_MED.render(f"Brinjët: {len(edges)}", True, TEXT_C)
    screen.blit(s, (px, py)); py += 20
    s = F_MED.render(f"Burimi : {src_node if src_node is not None else '—'}", True, NODE_SRC)
    screen.blit(s, (px, py)); py += 20
    s = F_MED.render(f"Dest.  : {dst_node if dst_node is not None else '—'}", True, NODE_DST)
    screen.blit(s, (px, py)); py += 24

    pygame.draw.line(screen, (40,60,90),(px,py),(W-10,py)); py+=8

    if step_msg:
        words = step_msg.split()
        line, lines2 = "", []
        for w in words:
            if len(line)+len(w)+1 > 20:
                lines2.append(line); line = w
            else:
                line = (line+" "+w).strip()
        if line: lines2.append(line)
        for ln in lines2:
            s2 = F_SML.render(ln, True, (220,240,180))
            screen.blit(s2, (px,py)); py += 17
    py += 6

    if path and mode == "result":
        pygame.draw.line(screen, (40,60,90),(px,py),(W-10,py)); py+=8
        s3 = F_MED.render("Rruga:", True, GOLD)
        screen.blit(s3, (px,py)); py+=20
        for i, nid in enumerate(path):
            arrow = "→ " if i>0 else "✦ "
            s4 = F_SML.render(arrow+str(nid), True, PATH_C)
            screen.blit(s4, (px,py)); py+=17
        if dst_node in dist_result:
            tot = F_MED.render(f"Cost: {dist_result[dst_node]}", True, GREEN)
            screen.blit(tot, (px,py))

    if has_negative():
        pw = F_SML.render("⚠ Peshë negative!", True, NEG_C)
        screen.blit(pw, (px, H-30))

    esc = F_SML.render("ESC = dil", True, (80,90,110))
    screen.blit(esc, (GRAPH_W+10, H-20))

    pygame.display.flip()

def find_node(mx, my):
    for nid, pos in nodes.items():
        if math.hypot(mx-pos[0], my-pos[1]) < 22:
            return nid
    return None

running = True
while running:
    dt = clock.tick(60)

    if error_timer > 0:
        error_timer -= dt
        if error_timer <= 0:
            error_msg = ""

    for event in pygame.event.get():
        if event.type == pygame.QUIT or (
                event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE):
            running = False

        elif event.type == pygame.KEYDOWN:
            if input_mode:
                if event.key == pygame.K_RETURN:
                    try:
                        w = int(input_buf)
                        edges.append((pending_edge[0], pending_edge[1], w))
                        neg_warning = has_negative()
                    except:
                        error_msg = "Vlera e pavlefshme!"; error_timer = 2000
                    input_mode = False; input_buf = ""; pending_edge = None
                elif event.key == pygame.K_BACKSPACE:
                    input_buf = input_buf[:-1]
                elif event.unicode in "0123456789-":
                    input_buf += event.unicode
            else:
                if event.key == pygame.K_1:   mode = "add_node"; sel_node=None
                elif event.key == pygame.K_2: mode = "add_edge"; sel_node=None
                elif event.key == pygame.K_3: mode = "set_src";  sel_node=None
                elif event.key == pygame.K_4: mode = "set_dst";  sel_node=None
                elif event.key == pygame.K_r:
                    nodes.clear(); edges.clear()
                    node_id=0; mode="add_node"; sel_node=None
                    src_node=dst_node=None; neg_warning=False
                    path=[]; dist_result={}; vis_result=set()
                    step_steps=[]; step_idx=0; step_dist={}; step_vis=set()
                    step_msg=""; error_msg=""
                elif event.key == pygame.K_s:
                    if src_node is not None and dst_node is not None and len(nodes)>1:
                        adj = get_adj()
                        neg_warning = has_negative()
                        if neg_warning:
                            step_msg = "⚠ Peshë negative! Dijkstra mund të japë rezultat të gabuar."
                        step_steps, path, dist_result = dijkstra(src_node, dst_node, adj)
                        step_idx = 0; step_dist = {n: float('inf') for n in nodes}
                        step_dist[src_node] = 0; step_vis = set()
                        mode = "result"
                        step_msg = "Gati! Shtyp N për hap-pas-hapi."
                    else:
                        error_msg = "Cakto burimin dhe destinacionin!"; error_timer=2000
                elif event.key == pygame.K_n and mode == "result":
                    if step_idx < len(step_steps):
                        s = step_steps[step_idx]
                        if s[0] == "visit":
                            _, u, d, vis = s
                            step_dist = d; step_vis = vis
                            step_msg = f"Vizitoj nyjen {u} (dist={d[u]})"
                        else:
                            _, u, v, d, vis = s
                            step_dist = d; step_vis = vis
                            step_msg = f"Relaksoj {u}→{v}: dist[{v}]={d[v]}"
                        step_idx += 1
                    else:
                        step_msg = "Dijkstra përfundoi!"

        elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1 and not input_mode:
            mx, my = event.pos
            if mx >= GRAPH_W: continue
            clicked = find_node(mx, my)

            if mode == "add_node":
                nodes[node_id] = (mx, my)
                node_id += 1
            elif mode == "add_edge":
                if clicked is not None:
                    if sel_node is None:
                        sel_node = clicked
                    elif sel_node != clicked:
                        # check duplicate
                        exists = any((u==sel_node and v==clicked) or (u==clicked and v==sel_node)
                                     for u,v,_ in edges)
                        if not exists:
                            pending_edge = (sel_node, clicked)
                            input_mode = True; input_buf = ""
                        sel_node = None
                    else:
                        sel_node = None
            elif mode == "set_src":
                if clicked is not None: src_node = clicked
            elif mode == "set_dst":
                if clicked is not None: dst_node = clicked

        elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 3:
            mx, my = event.pos
            clicked = find_node(mx, my)
            if clicked is not None and not input_mode:
                del nodes[clicked]
                edges[:] = [(u,v,w) for u,v,w in edges if u!=clicked and v!=clicked]
                if src_node == clicked: src_node = None
                if dst_node == clicked: dst_node = None
                neg_warning = has_negative()

    draw_scene()

pygame.quit()
sys.exit()


SystemExit: 

---

## Ushtrimi 3 — Bellman-Ford: Rrjeti i Dorëzimit

### Teoria: Bellman-Ford me peshe negative

Imagjino një **rrjet dorëzimi** ku disa rrugë kanë **kosto negative** (subvencione qeveritare ose zbritje). Dijkstra nuk mund t'i trajtojë. **Bellman-Ford** e zgjidh këtë!

**Relaksimi i brinjëve:**
```
Nëse dist[u] + w(u,v) < dist[v]:
    dist[v] = dist[u] + w(u,v)
```
Kjo bëhet **V-1 herë** (V = numri i kulmeve). Pas kësaj, nëse mund të relaksojmë akoma → **cikël negativ** (distancat mund të ulen pafundësisht).

### Animacioni

Rrjeti ka 8 nyje. Brinjët **të kuqe** = peshe negative (subvencione). Kamioni animohet gjatë rrugës optimale.

**Kontrollet:**
- `SPACE` — hap tjetër  `A` — auto-luaj
- `T` — nis kamionin drejt Hub
- `N` — shto cikël negativ (shiko detektimin!)
- `R` — rifillo | `ESC` — dil

In [4]:
import pygame, math, sys, time

pygame.init()
W, H = 1280, 720
screen = pygame.display.set_mode((0, 0), pygame.FULLSCREEN)
W, H = screen.get_size()
pygame.display.set_caption("Ushtrimi 3 — Bellman-Ford: Rrjeti i Dorëzimit")
clock = pygame.time.Clock()

BG       = (8, 14, 28)
PANEL_BG = (14, 20, 40)
NODE_C   = (50, 100, 180)
NODE_VIS = (50, 180, 80)
NODE_SRC = (255, 100, 50)
EDGE_C   = (55, 70, 105)
EDGE_REL = (120, 180, 255)
NEG_EDGE = (220, 80, 80)
PATH_C   = (255, 215, 0)
CYCLE_C  = (255, 40, 40)
TEXT_C   = (210, 225, 255)
GOLD     = (255, 215, 0)
WHITE    = (255, 255, 255)
GREEN    = (70, 200, 100)
RED      = (220, 60, 60)

F_TITLE = pygame.font.SysFont("consolas", 24, bold=True)
F_BIG   = pygame.font.SysFont("consolas", 18, bold=True)
F_MED   = pygame.font.SysFont("consolas", 15)
F_SML   = pygame.font.SysFont("consolas", 13)

# ── delivery network ─────────────────────────────────────────────────────────
NODES = {
    "Depot":  (130, 340),
    "Alfa":   (290, 160),
    "Beta":   (290, 340),
    "Gama":   (290, 520),
    "Delta":  (490, 240),
    "Eta":    (490, 440),
    "Zeta":   (680, 340),
    "Hub":    (820, 340),
}

EDGES_BASE = [
    ("Depot",  "Alfa",   6),
    ("Depot",  "Beta",   7),
    ("Depot",  "Gama",   9),
    ("Alfa",   "Delta",  5),
    ("Beta",   "Delta",  8),
    ("Beta",   "Eta",    -3),   # subvencion (negative)
    ("Gama",   "Eta",    7),
    ("Delta",  "Zeta",   -4),   # subvencion (negative)
    ("Eta",    "Zeta",   6),
    ("Zeta",   "Hub",    3),
    ("Delta",  "Eta",    4),
]

source_node = "Depot"
GRAPH_W = 900

# ── bellman-ford ─────────────────────────────────────────────────────────────
def bellman_ford(nodes_list, edges_list, src):
    dist = {n: float('inf') for n in nodes_list}
    prev = {n: None for n in nodes_list}
    dist[src] = 0
    n = len(nodes_list)
    steps = []   # (iteration, u, v, w, dist_snapshot, relaxed)
    neg_cycle_edges = []

    for i in range(n - 1):
        relaxed_any = False
        for u, v, w in edges_list:
            if dist[u] != float('inf') and dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                prev[v] = u
                relaxed_any = True
                steps.append(("relax", i+1, u, v, w, dict(dist), (u,v)))
        steps.append(("iter_end", i+1, dict(dist)))
        if not relaxed_any:
            break

    # check negative cycle
    for u, v, w in edges_list:
        if dist[u] != float('inf') and dist[u] + w < dist[v]:
            neg_cycle_edges.append((u, v))

    return steps, dist, prev, neg_cycle_edges

# ── truck animation ──────────────────────────────────────────────────────────
def get_path(prev, dst):
    path = []
    cur = dst
    while cur is not None:
        path.append(cur)
        cur = prev.get(cur)
    path.reverse()
    return path

def lerp(a, b, t):
    return (a[0]+(b[0]-a[0])*t, a[1]+(b[1]-a[1])*t)

# ── state ─────────────────────────────────────────────────────────────────────
edges = list(EDGES_BASE)
steps, dist_final, prev_final, neg_cycles = bellman_ford(list(NODES.keys()), edges, source_node)
step_idx  = 0
step_dist = {n: float('inf') for n in NODES}; step_dist[source_node] = 0
relaxed_edge = None
iter_num  = 0
phase     = "ready"   # ready | stepping | truck | done
truck_path= []
truck_seg = 0
truck_t   = 0.0
truck_dst = "Hub"
step_msg  = ""
neg_cycle_mode = False
auto_play = False
last_auto = 0.0

def draw_arrow_edge(u, v, col, width=2, offset=0):
    p1, p2 = list(NODES[u]), list(NODES[v])
    # small offset for parallel edges
    dx, dy = p2[0]-p1[0], p2[1]-p1[1]
    length = math.hypot(dx,dy)
    if length<1: return
    nx, ny = -dy/length*offset, dx/length*offset
    sp = (p1[0]+nx, p1[1]+ny)
    ep = (p2[0]+nx, p2[1]+ny)
    pygame.draw.line(screen, col, (int(sp[0]),int(sp[1])), (int(ep[0]),int(ep[1])), width)
    # arrowhead
    ux,uy = (ep[0]-sp[0])/length, (ep[1]-sp[1])/length
    ax = ep[0]-ux*12-uy*6; ay = ep[1]-uy*12+ux*6
    bx = ep[0]-ux*12+uy*6; by = ep[1]-uy*12-ux*6
    pygame.draw.polygon(screen, col, [(int(ep[0]),int(ep[1])),
                                       (int(ax),int(ay)),(int(bx),int(by))])

def draw_truck(pos, color=(255,160,40)):
    x,y = int(pos[0]), int(pos[1])
    # truck body
    pygame.draw.rect(screen, color, (x-18,y-10,36,20), border_radius=4)
    pygame.draw.rect(screen, (200,220,255),(x+2,y-14,12,10), border_radius=2)
    for wx in [x-12, x+8]:
        pygame.draw.circle(screen, (30,30,30),(wx,y+10),5)
        pygame.draw.circle(screen, (160,160,160),(wx,y+10),3)

def reset_steps():
    global steps, dist_final, prev_final, neg_cycles, step_idx, step_dist, relaxed_edge
    global iter_num, phase, truck_path, truck_seg, truck_t, step_msg, auto_play
    steps, dist_final, prev_final, neg_cycles = bellman_ford(list(NODES.keys()), edges, source_node)
    step_idx = 0
    step_dist = {n: float('inf') for n in NODES}
    step_dist[source_node] = 0
    relaxed_edge = None; iter_num = 0; phase = "ready"
    truck_path=[]; truck_seg=0; truck_t=0.0
    step_msg="Shtyp SPACE për hap-pas-hapi ose A për auto-luaj"; auto_play=False

reset_steps()

def draw_scene():
    screen.fill(BG)
    pygame.draw.rect(screen, PANEL_BG,(GRAPH_W,0,W-GRAPH_W,H))
    pygame.draw.line(screen,(40,60,90),(GRAPH_W,0),(GRAPH_W,H),2)

    path_set = set()
    if truck_path:
        for i in range(len(truck_path)-1):
            path_set.add((truck_path[i],truck_path[i+1]))

    # draw edges
    for u,v,w in edges:
        in_path = (u,v) in path_set
        is_neg  = w < 0
        in_cycle = neg_cycle_mode and any((u==a and v==b) for a,b in neg_cycles)
        is_relax = relaxed_edge == (u,v)
        if in_cycle: col = CYCLE_C; lw = 4
        elif in_path: col = PATH_C; lw = 4
        elif is_relax: col = EDGE_REL; lw = 3
        elif is_neg: col = NEG_EDGE; lw = 2
        else: col = EDGE_C; lw = 2
        draw_arrow_edge(u, v, col, lw)
        mx = (NODES[u][0]+NODES[v][0])//2
        my = (NODES[u][1]+NODES[v][1])//2
        wlbl = F_SML.render(str(w), True, NEG_EDGE if is_neg else (180,200,240))
        screen.blit(wlbl,(mx+4,my-10))

    # nodes
    for name, pos in NODES.items():
        col = NODE_SRC if name == source_node else (
              NODE_VIS if step_dist.get(name,float('inf')) < float('inf') else NODE_C)
        r = 22
        pygame.draw.circle(screen, col, pos, r)
        pygame.draw.circle(screen, WHITE, pos, r, 2)
        lbl = F_SML.render(name, True, WHITE)
        screen.blit(lbl,(pos[0]-lbl.get_width()//2, pos[1]-lbl.get_height()//2))
        d = step_dist.get(name, float('inf'))
        ds = "∞" if d==float('inf') else str(d)
        dl = F_SML.render(ds, True, GOLD)
        screen.blit(dl,(pos[0]-dl.get_width()//2, pos[1]+26))

    # truck
    if phase in ("truck","done") and truck_path:
        if 0 <= truck_seg < len(truck_path)-1:
            p1 = NODES[truck_path[truck_seg]]
            p2 = NODES[truck_path[truck_seg+1]]
            pos = lerp(p1,p2,truck_t)
        else:
            pos = NODES[truck_path[-1]]
        draw_truck(pos)

    # negative cycle alert
    if neg_cycle_mode and neg_cycles:
        ban = pygame.Surface((GRAPH_W,50),pygame.SRCALPHA)
        ban.fill((200,20,20,180))
        screen.blit(ban,(0,H-52))
        t = F_BIG.render("🚨  CIKËL NEGATIV u zbulua! Bellman-Ford e detekton këtë rast!", True, WHITE)
        screen.blit(t,(GRAPH_W//2-t.get_width()//2,H-44))

    # iter counter
    if iter_num > 0:
        it = F_MED.render(f"Iterimi: {iter_num} / {len(NODES)-1}", True,(160,200,255))
        screen.blit(it,(10,10))

    # right panel
    px,py = GRAPH_W+10, 10
    title = F_TITLE.render("Bellman-Ford", True, GOLD)
    screen.blit(title,(px,py)); py+=36

    controls = [
        ("SPACE","Hap tjetër"),
        ("A",    "Auto-luaj"),
        ("T",    "Kamion→Hub"),
        ("N",    "Shto cikël negativ"),
        ("R",    "Rifillo"),
    ]
    for k,v in controls:
        s = F_SML.render(f"[{k}] {v}", True, TEXT_C)
        screen.blit(s,(px,py)); py+=18
    py+=10

    pygame.draw.line(screen,(40,60,90),(px,py),(W-10,py)); py+=8

    s0 = F_MED.render(f"Burimi: {source_node}", True, NODE_SRC)
    screen.blit(s0,(px,py)); py+=22

    for name in NODES:
        d = step_dist.get(name,float('inf'))
        ds = "∞" if d==float('inf') else str(d)
        col = GOLD if name==truck_dst else TEXT_C
        row = F_SML.render(f"  {name:<8} {ds:>5}", True, col)
        screen.blit(row,(px,py)); py+=17
    py+=8

    pygame.draw.line(screen,(40,60,90),(px,py),(W-10,py)); py+=8

    if step_msg:
        words = step_msg.split()
        line,lines2="",[]
        for w in words:
            if len(line)+len(w)+1>22: lines2.append(line); line=w
            else: line=(line+" "+w).strip()
        if line: lines2.append(line)
        for ln in lines2:
            s2=F_SML.render(ln,True,(220,240,180))
            screen.blit(s2,(px,py)); py+=17

    if neg_cycles:
        wn=F_SML.render("⚠ Cikël negativ!",True,RED)
        screen.blit(wn,(px,H-30))

    esc=F_SML.render("ESC = dil",True,(80,90,110))
    screen.blit(esc,(GRAPH_W+10,H-20))

    pygame.display.flip()

def advance_step():
    global step_idx, step_dist, relaxed_edge, iter_num, step_msg, phase
    if step_idx >= len(steps):
        phase = "done"
        step_msg = "Bellman-Ford përfundoi! Shtyp T për kamionin."
        return
    s = steps[step_idx]
    if s[0] == "relax":
        _, it, u, v, w, d, re = s
        step_dist = d; relaxed_edge = re; iter_num = it
        step_msg = f"It.{it}: Relaksoj {u}→{v} (w={w}) → dist[{v}]={d[v]}"
    else:
        _, it, d = s
        step_dist = d; relaxed_edge = None; iter_num = it
        step_msg = f"Iterimi {it} përfundoi."
    step_idx += 1

running = True
while running:
    dt = clock.tick(60)/1000.0

    for event in pygame.event.get():
        if event.type == pygame.QUIT or (
                event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE):
            running = False
        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_SPACE:
                advance_step()
                auto_play = False
            elif event.key == pygame.K_a:
                auto_play = not auto_play
                step_msg = "Auto: " + ("ON" if auto_play else "OFF")
            elif event.key == pygame.K_t:
                path_ = get_path(prev_final, truck_dst)
                if len(path_) > 1:
                    truck_path = path_; truck_seg=0; truck_t=0.0; phase="truck"
                    step_msg = f"Kamioni lëviz: {' → '.join(path_)}"
            elif event.key == pygame.K_n:
                # add a negative cycle
                neg_cycle_mode = True
                if ("Delta","Beta",-10) not in edges:
                    edges.append(("Delta","Beta",-10))
                if ("Beta","Eta",-8) not in [(u,v,w) for u,v,w in edges if w==-8]:
                    pass
                reset_steps()
                step_msg = "Cikël negativ u shtua! Ri-ekzekuton..."
            elif event.key == pygame.K_r:
                edges[:] = list(EDGES_BASE)
                neg_cycle_mode = False
                reset_steps()

    if auto_play and phase in ("ready","stepping"):
        now = time.time()
        if now - last_auto > 0.4:
            last_auto = now
            advance_step()

    # truck movement
    if phase == "truck" and truck_path:
        SPEED = 0.6
        truck_t += dt * SPEED
        if truck_t >= 1.0:
            truck_t -= 1.0; truck_seg += 1
            if truck_seg >= len(truck_path)-1:
                phase = "done"
                step_msg = f"Arritur Hub! Kosto totale: {dist_final.get(truck_dst,'?')}"

    draw_scene()

pygame.quit()
sys.exit()


SystemExit: 

---

## Ushtrimi 4 — Bellman-Ford Kuiz

### Teoria: Konceptet kryesore të Bellman-Ford

Ky kuiz interaktiv teston njohuritë tuaja mbi Bellman-Ford duke u fokusuar në:
- **Numrin e iterimeve** dhe distancat pas çdo iterimi
- **Detektimin e cikleve negative** (pasoja kryesore e Bellman-Ford)
- **Krahasimin** me Dijkstrën

**Grafu i kuizit:**  6 kulme (A–F), brinjë të drejtëzuara me peshe **pozitive dhe negative**.

### Loja

8 pyetje me zgjedhje të shumëfishtë. Çdo pyetje ka **timer 20 sekondash** — përgjigju shpejt për bonus pikë! Lexo **shpjegimin** pas çdo përgjigje.

**Klikoni opsionin tuaj!** | `ESC` → dil

In [ ]:
import pygame, sys, random

pygame.init()
W, H = 1280, 720
screen = pygame.display.set_mode((0, 0), pygame.FULLSCREEN)
W, H = screen.get_size()
pygame.display.set_caption("Ushtrimi 4 — Bellman-Ford: Kuiz Interaktiv")
clock = pygame.time.Clock()

BG       = (8,  14,  28)
PANEL    = (15, 22,  45)
GOLD     = (255, 215,  0)
WHITE    = (255, 255, 255)
GREEN    = (50,  200, 100)
RED      = (220,  60,  60)
BLUE     = (60,  130, 220)
GRAY     = (80,  90, 110)
TEXT_C   = (210, 225, 255)
BTN_NOR  = (30,  50,  90)
BTN_HVR  = (50,  80, 140)
BTN_COR  = (20,  90,  40)
BTN_WRG  = (90,  20,  20)

F_TITLE = pygame.font.SysFont("consolas", 30, bold=True)
F_BIG   = pygame.font.SysFont("consolas", 22, bold=True)
F_MED   = pygame.font.SysFont("consolas", 17)
F_SML   = pygame.font.SysFont("consolas", 14)

# ── quiz graph (visual) ──────────────────────────────────────────────────────
GNODES = {
    "A": (160, 340), "B": (320, 180), "C": (320, 500),
    "D": (520, 180), "E": (520, 500), "F": (700, 340),
}
GEDGES = [
    ("A","B",4), ("A","C",2),
    ("B","D",5), ("B","C",-1),
    ("C","E",3),
    ("D","F",2), ("D","E",1),
    ("E","F",4),
    ("B","E",-3),
]

def bellman_ford_full(nodes, edges, src):
    dist = {n: float('inf') for n in nodes}
    prev = {n: None for n in nodes}
    dist[src] = 0
    n = len(nodes)
    snap = [dict(dist)]
    for _ in range(n-1):
        changed = False
        for u,v,w in edges:
            if dist[u] != float('inf') and dist[u]+w < dist[v]:
                dist[v] = dist[u]+w; prev[v] = u; changed = True
        snap.append(dict(dist))
        if not changed: break
    # negative cycle check
    neg = False
    for u,v,w in edges:
        if dist[u] != float('inf') and dist[u]+w < dist[v]:
            neg = True; break
    return dist, snap, neg

nodes_list = list(GNODES.keys())
dist_A, snaps_A, has_neg_A = bellman_ford_full(nodes_list, GEDGES, "A")
dist_B, snaps_B, has_neg_B = bellman_ford_full(nodes_list, GEDGES, "B")

def fmt(d): return "∞" if d==float('inf') else str(d)

QUESTIONS = [
    {
        "q": "Pas iterimit të parë të Bellman-Ford\n(burim A), sa është dist[D]?",
        "opts": ["9","5","∞","4"],
        "ans": 0,
        "explain": "A→B(4)+B→D(5)=9. Iterimi 1 relakson këto dy rrugë."
    },
    {
        "q": "Cila është distanca finale nga A në F\nme Bellman-Ford?",
        "opts": [str(dist_A.get('F','∞')), "11","13","15"],
        "ans": 0,
        "explain": f"A→B→E→D→F = 4+(-3)+1+2 = {dist_A.get('F','?')} (rruga optimale)"
    },
    {
        "q": "Sa iterime maksimale bën Bellman-Ford\nnë një graf me V kulme?",
        "opts": ["V-1","V","V+1","V²"],
        "ans": 0,
        "explain": "Bellman-Ford ekzekuton V-1 iterime (rruga më e gjatë ka V-1 brinjë)."
    },
    {
        "q": "Bellman-Ford mund të zbulojë\nciklet negative?",
        "opts": ["Po","Jo","Vetëm me peshë pozitive","Vetëm grafe të pa-drejtimit"],
        "ans": 0,
        "explain": "Po! Nëse pas V-1 iterimeve, ende mund të relaksosh, ka cikël negativ."
    },
    {
        "q": "Cili është kompleksiteti kohor\ni Bellman-Ford?",
        "opts": ["O(VE)","O(V²)","O(E log V)","O(V+E)"],
        "ans": 0,
        "explain": "O(VE): V-1 iterime × E brinjë secila iterim."
    },
    {
        "q": "Dijkstra vs Bellman-Ford:\ncili mbështet peshë negative?",
        "opts": ["Vetëm Bellman-Ford","Vetëm Dijkstra","Të dy","Asnjë"],
        "ans": 0,
        "explain": "Dijkstra NËNKUPTON peshe ≥ 0. Bellman-Ford funksionon me negative."
    },
    {
        "q": "Pas iterimit 2 (burim A),\nsa është dist[E]?",
        "opts": [fmt(snaps_A[2].get('E',float('inf'))), "7","5","3"],
        "ans": 0,
        "explain": f"A→B→E = 4+(-3) = 1, ose A→C→E = 2+3 = 5. Min = {fmt(snaps_A[2].get('E',float('inf')))}."
    },
    {
        "q": "Bellman-Ford zbatohet si:\nburim i vetëm apo të gjitha çiftet?",
        "opts": ["Burim i vetëm","Të gjitha çiftet","Të dyja","Asnjë"],
        "ans": 0,
        "explain": "Bellman-Ford gjen rrugët nga NJË burim tek të gjitha kulmet tjera."
    },
]

random.shuffle(QUESTIONS)
QUESTIONS = QUESTIONS[:6]

# ── state ─────────────────────────────────────────────────────────────────────
q_idx     = 0
score     = 0
selected  = None
answered  = False
show_expl = False
game_over = False
hover_btn = -1
time_left = 20.0
time_bonus= True
GRAPH_W   = 780

def draw_graph(highlight_dist=None):
    # background
    pygame.draw.rect(screen, (12,18,35), (0,0,GRAPH_W,H))

    for u,v,w in GEDGES:
        p1,p2 = GNODES[u],GNODES[v]
        neg = w < 0
        col = (220,80,80) if neg else (60,80,120)
        pygame.draw.line(screen,col,p1,p2,3)
        mx,my = (p1[0]+p2[0])//2,(p1[1]+p2[1])//2
        wl = F_SML.render(str(w),True,col)
        screen.blit(wl,(mx+4,my-10))
        # arrow
        dx,dy = p2[0]-p1[0],p2[1]-p1[1]
        length = (dx**2+dy**2)**0.5
        if length>1:
            ux,uy = dx/length,dy/length
            ex,ey = p2[0]-ux*24,p2[1]-uy*24
            ax=ex-ux*10-uy*6; ay=ey-uy*10+ux*6
            bx=ex-ux*10+uy*6; by=ey-uy*10-ux*6
            pygame.draw.polygon(screen,col,[(int(ex),int(ey)),(int(ax),int(ay)),(int(bx),int(by))])

    for name,pos in GNODES.items():
        col = (255,100,50) if name=="A" else (50,110,200)
        pygame.draw.circle(screen,col,pos,22)
        pygame.draw.circle(screen,WHITE,pos,22,2)
        lbl = F_MED.render(name,True,WHITE)
        screen.blit(lbl,(pos[0]-lbl.get_width()//2,pos[1]-lbl.get_height()//2))
        if highlight_dist:
            d = highlight_dist.get(name,float('inf'))
            ds = "∞" if d==float('inf') else str(d)
            dl = F_SML.render(ds,True,GOLD)
            screen.blit(dl,(pos[0]-dl.get_width()//2,pos[1]+26))

    # source label
    sl = F_SML.render("Burimi: A",True,(200,180,100))
    screen.blit(sl,(10,10))

def get_btn_rect(i, total=4):
    col_w = (GRAPH_W-40)//2
    col   = i % 2
    row   = i // 2
    x = 20 + col*(col_w+10)
    y = H - 220 + row*90
    return pygame.Rect(x, y, col_w, 75)

running = True
while running:
    dt = clock.tick(60)/1000.0

    if not game_over and not answered:
        time_left -= dt
        if time_left <= 0:
            time_left = 0
            answered  = True
            selected  = -1  # timeout
            show_expl = True

    for event in pygame.event.get():
        if event.type == pygame.QUIT or (
                event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE):
            running = False

        elif event.type == pygame.MOUSEMOTION and not answered:
            mx,my = event.pos
            hover_btn = -1
            if not game_over:
                for i in range(4):
                    if get_btn_rect(i).collidepoint(mx,my):
                        hover_btn = i; break

        elif event.type == pygame.MOUSEBUTTONDOWN and event.button == 1:
            mx,my = event.pos

            if game_over:
                # restart button
                rb = pygame.Rect(W//2-100,H//2+100,200,50)
                if rb.collidepoint(mx,my):
                    random.shuffle(QUESTIONS); QUESTIONS2 = QUESTIONS[:6]
                    q_idx=0; score=0; selected=None; answered=False
                    show_expl=False; game_over=False; time_left=20.0
                continue

            if answered:
                # next button
                nb = pygame.Rect(GRAPH_W+20,H-70,W-GRAPH_W-40,50)
                if nb.collidepoint(mx,my):
                    q_idx += 1
                    if q_idx >= len(QUESTIONS):
                        game_over = True
                    else:
                        answered=False; selected=None; show_expl=False; time_left=20.0
            else:
                for i in range(4):
                    if get_btn_rect(i).collidepoint(mx,my):
                        selected = i; answered = True; show_expl = True
                        correct = QUESTIONS[q_idx]["ans"]
                        bonus = max(0, int(time_left*0.5)) if time_bonus else 0
                        if i == correct:
                            score += 10 + bonus
                        break

    # ── draw ──────────────────────────────────────────────────────────────────
    screen.fill(BG)

    if game_over:
        screen.fill((8,12,25))
        tl = F_TITLE.render("LOJA MBAROI!", True, GOLD)
        screen.blit(tl,(W//2-tl.get_width()//2, H//2-120))
        sc = F_BIG.render(f"Rezultati: {score} / {len(QUESTIONS)*10}", True, WHITE)
        screen.blit(sc,(W//2-sc.get_width()//2,H//2-60))
        star = "⭐" * min(3, score//(len(QUESTIONS)*10//3+1))
        sm = F_BIG.render(star or "Provo Sërish!", True, GOLD)
        screen.blit(sm,(W//2-sm.get_width()//2,H//2))
        rb = pygame.Rect(W//2-100,H//2+100,200,50)
        pygame.draw.rect(screen,BLUE,rb,border_radius=10)
        rt = F_MED.render("Luaj Sërish",True,WHITE)
        screen.blit(rt,(rb.centerx-rt.get_width()//2,rb.centery-rt.get_height()//2))
        pygame.display.flip(); clock.tick(60); continue

    # draw graph
    draw_graph(dist_A if answered else None)

    # separator
    pygame.draw.line(screen,(40,60,90),(GRAPH_W,0),(GRAPH_W,H),2)
    pygame.draw.rect(screen,PANEL,(GRAPH_W,0,W-GRAPH_W,H))

    # right panel
    px,py = GRAPH_W+15, 12
    title = F_TITLE.render("Bellman-Ford Kuiz", True, GOLD)
    screen.blit(title,(px,py)); py+=40

    # score + time
    sc_s = F_BIG.render(f"Pikët: {score}", True, GOLD)
    screen.blit(sc_s,(px,py))
    q_prog = F_MED.render(f"Pyetja {q_idx+1}/{len(QUESTIONS)}", True, (160,180,220))
    screen.blit(q_prog,(W-q_prog.get_width()-10,py))
    py += 30

    # timer bar
    bar_w = W-GRAPH_W-30
    pygame.draw.rect(screen,(30,40,60),(px,py,bar_w,14),border_radius=7)
    frac = time_left/20.0
    col_t = GREEN if frac>0.5 else (GOLD if frac>0.25 else RED)
    pygame.draw.rect(screen,col_t,(px,py,int(bar_w*frac),14),border_radius=7)
    py += 22

    # question text
    pygame.draw.line(screen,(40,60,90),(px,py),(W-10,py)); py+=8
    qtext = QUESTIONS[q_idx]["q"]
    for line in qtext.split('\n'):
        ql = F_MED.render(line, True, WHITE)
        screen.blit(ql,(px,py)); py+=24
    py += 8

    # answer buttons on left panel bottom
    qd = QUESTIONS[q_idx]
    for i,opt in enumerate(qd["opts"]):
        r = get_btn_rect(i)
        if answered:
            if i == qd["ans"]: col = BTN_COR
            elif i == selected: col = BTN_WRG
            else: col = BTN_NOR
        else:
            col = BTN_HVR if hover_btn==i else BTN_NOR
        pygame.draw.rect(screen,col,r,border_radius=10)
        pygame.draw.rect(screen,(80,100,150),r,2,border_radius=10)
        ltr = chr(65+i)
        tl_b = F_MED.render(f"{ltr}) {opt}", True, WHITE)
        screen.blit(tl_b,(r.x+12, r.centery-tl_b.get_height()//2))

    # explanation
    if show_expl:
        expl = QUESTIONS[q_idx]["explain"]
        words = expl.split(); line=""; lines=[]
        for w in words:
            if len(line)+len(w)+1>28: lines.append(line); line=w
            else: line=(line+" "+w).strip()
        if line: lines.append(line)
        for ln in lines:
            el = F_SML.render(ln,True,(200,230,180))
            screen.blit(el,(px,py)); py+=17
        # next button
        nb = pygame.Rect(px,H-65,W-GRAPH_W-30,48)
        pygame.draw.rect(screen,BLUE,nb,border_radius=10)
        nt = F_MED.render("Tjetër →" if q_idx<len(QUESTIONS)-1 else "Rezultatet!",True,WHITE)
        screen.blit(nt,(nb.centerx-nt.get_width()//2,nb.centery-nt.get_height()//2))

    esc = F_SML.render("ESC = dil",True,(80,90,110))
    screen.blit(esc,(GRAPH_W+10,H-20))

    pygame.display.flip()

pygame.quit()
sys.exit()


SystemExit: 

---

## Ushtrimi 5 — Floyd-Warshall: Matrica e Distancave

### Teoria: Floyd-Warshall dhe programimi dinamik

**Floyd-Warshall** gjen distancat midis **të gjitha çifteve** të kulmeve me një qasje programim-dinamik:

```
për k = 0 deri N-1:
    për i = 0 deri N-1:
        për j = 0 deri N-1:
            D[i][j] = min(D[i][j], D[i][k] + D[k][j])
```

Ku **k** është kulmi **ndërmjetës** i provuar. Algoritmi provon sistematikisht të gjitha kulmet si ndërmjetës.

### Animacioni — Ekran i ndarë

**Majtas:** Grafi me 6 qytete (Alfa–Zeta).  
**Djathtas:** Matrica D[i][j] që përditësohet **live** — qelizat që ndryshojnë bëhen të gjelbra.

Pas ekzekutimit, klikoni **2 nyje** për të shfaqur rrugën optimale midis tyre.

**Kontrollet:**
- `SPACE` — hap tjetër  `A` — auto-luaj  
- `+/-` — shpejtësi animimi
- Klik 2 nyje → shfaq rrugën optimale
- `R` — rifillo | `ESC` — dil

In [14]:
import pygame, math, sys, time

pygame.init()
W, H = 1400, 780
screen = pygame.display.set_mode((0, 0), pygame.FULLSCREEN)
W, H = screen.get_size()
pygame.display.set_caption("Ushtrimi 5 — Floyd-Warshall: Matrica e Distancave")
clock = pygame.time.Clock()

BG       = (8,  14,  28)
PANEL_BG = (14, 22,  42)
NODE_C   = (50, 100, 190)
NODE_K   = (220, 160,  30)  # intermediate k
NODE_I   = (200,  80,  50)  # row i
NODE_J   = (50,  180, 180)  # col j
EDGE_C   = (55,  70, 110)
PATH_C   = (255, 215,   0)
TEXT_C   = (210, 225, 255)
GOLD     = (255, 215,   0)
WHITE    = (255, 255, 255)
GREEN    = (60,  210, 100)
RED      = (220,  60,  60)
CELL_DEF = (20,  30,  55)
CELL_K   = (80,  60,  20)
CELL_I   = (70,  25,  25)
CELL_J   = (20,  60,  70)
CELL_UPD = (30,  90,  40)
CELL_HL  = (40, 120, 160)

F_TITLE = pygame.font.SysFont("consolas", 22, bold=True)
F_BIG   = pygame.font.SysFont("consolas", 17, bold=True)
F_MED   = pygame.font.SysFont("consolas", 15)
F_SML   = pygame.font.SysFont("consolas", 13)

INF = float('inf')

CITIES = ["Alfa","Beta","Gama","Delta","Eta","Zeta"]
N = len(CITIES)

# adjacency matrix (INF = no direct edge)
ADJ = [
    [0,   3,   INF, 7,   INF, INF],
    [3,   0,   2,   INF, 4,   INF],
    [INF, 2,   0,   INF, 1,   5  ],
    [7,   INF, INF, 0,   2,   INF],
    [INF, 4,   1,   2,   0,   3  ],
    [INF, INF, 5,   INF, 3,   0  ],
]

GRAPH_POS = {
    "Alfa":  (150, 200),
    "Beta":  (330, 120),
    "Gama":  (500, 200),
    "Delta": (150, 480),
    "Eta":   (330, 560),
    "Zeta":  (500, 480),
}
GRAPH_W = 680
MAT_X   = GRAPH_W + 20
MAT_CELL = 62

def build_steps():
    D = [row[:] for row in ADJ]
    P = [[j if ADJ[i][j] < INF and i!=j else None for j in range(N)] for i in range(N)]
    steps = []
    for k in range(N):
        for i in range(N):
            for j in range(N):
                if D[i][k] < INF and D[k][j] < INF:
                    nd = D[i][k] + D[k][j]
                    if nd < D[i][j]:
                        D[i][j] = nd
                        P[i][j] = P[i][k]
                        steps.append(("update", k, i, j, nd, [row[:] for row in D]))
                    else:
                        steps.append(("check",  k, i, j, D[i][j], [row[:] for row in D]))
    return steps, [row[:] for row in D], P

steps, D_final, P_final = build_steps()

# initial matrix
D_cur = [row[:] for row in ADJ]

k_cur = i_cur = j_cur = -1
step_idx  = 0
phase     = "ready"
step_msg  = ""
auto_play = False
speed     = 0.08   # seconds between auto-steps
last_auto = 0.0
updated_cells = set()  # (i,j) recently updated
flash_timer   = {}     # (i,j) -> timer
sel_src = sel_dst = None  # for path highlight

def get_path(src, dst, P):
    if D_final[src][dst] == INF: return []
    path = [src]
    cur  = src
    while cur != dst:
        cur = P[cur][dst]
        if cur is None: return []
        path.append(cur)
    return path

def draw_graph(path_nodes=None):
    path_set = set()
    if path_nodes:
        for i2 in range(len(path_nodes)-1):
            u, v = path_nodes[i2], path_nodes[i2+1]
            path_set.add((u,v)); path_set.add((v,u))

    for i2,ci in enumerate(CITIES):
        for j2 in range(i2+1, N):
            cj = CITIES[j2]
            if ADJ[i2][j2] < INF:
                p1,p2 = GRAPH_POS[ci],GRAPH_POS[cj]
                in_path = (i2,j2) in path_set or (j2,i2) in path_set
                col = PATH_C if in_path else EDGE_C
                lw  = 4 if in_path else 2
                pygame.draw.line(screen,col,p1,p2,lw)
                mx,my=(p1[0]+p2[0])//2,(p1[1]+p2[1])//2
                wl=F_SML.render(str(ADJ[i2][j2]),True,(160,180,220))
                screen.blit(wl,(mx+3,my-8))

    for idx2,name in enumerate(CITIES):
        pos = GRAPH_POS[name]
        if idx2 == k_cur: col = NODE_K
        elif idx2 == i_cur: col = NODE_I
        elif idx2 == j_cur: col = NODE_J
        elif path_nodes and idx2 in path_nodes: col = (80,210,100)
        else: col = NODE_C
        pygame.draw.circle(screen,col,pos,22)
        pygame.draw.circle(screen,WHITE,pos,22,2)
        lbl=F_SML.render(name[:4],True,WHITE)
        screen.blit(lbl,(pos[0]-lbl.get_width()//2,pos[1]-lbl.get_height()//2))

def draw_matrix(D, path_nodes=None):
    path_pairs = set()
    if path_nodes:
        for i2 in range(len(path_nodes)-1):
            path_pairs.add((path_nodes[i2],path_nodes[i2+1]))
            path_pairs.add((path_nodes[i2+1],path_nodes[i2]))

    # header
    ht = F_BIG.render("Matrica D[i][j]", True, GOLD)
    screen.blit(ht,(MAT_X, 12))

    # column headers
    for j2,name in enumerate(CITIES):
        cx = MAT_X + 80 + j2*MAT_CELL
        ch = F_SML.render(name[:4], True, NODE_J if j2==j_cur else (160,180,220))
        screen.blit(ch,(cx+MAT_CELL//2-ch.get_width()//2, 40))

    for i2 in range(N):
        # row header
        rh = F_SML.render(CITIES[i2][:4], True, NODE_I if i2==i_cur else (160,180,220))
        screen.blit(rh,(MAT_X+2, 62+i2*MAT_CELL+MAT_CELL//2-rh.get_height()//2))

        for j2 in range(N):
            cx = MAT_X + 80 + j2*MAT_CELL
            cy = 62 + i2*MAT_CELL
            rect = pygame.Rect(cx, cy, MAT_CELL-2, MAT_CELL-2)

            # background color
            if i2==j_cur and j2==k_cur: bg=CELL_K
            elif i2==i_cur and j2==k_cur: bg=CELL_K
            elif i2==k_cur and j2==j_cur: bg=CELL_K
            elif i2==i_cur and j2==j2 and (i2,j2) in flash_timer: bg=CELL_UPD
            elif i2==i_cur and j2==j_cur: bg=CELL_I
            elif i2==k_cur: bg=CELL_K
            elif j2==k_cur: bg=CELL_K
            elif (i2,j2) in flash_timer: bg=CELL_UPD
            elif (i2,j2) in path_pairs: bg=(30,70,30)
            else: bg=CELL_DEF

            pygame.draw.rect(screen,bg,rect,border_radius=4)
            pygame.draw.rect(screen,(40,55,85),rect,1,border_radius=4)

            val = D[i2][j2]
            vs  = "∞" if val==INF else str(val)
            col = (80,220,80) if (i2,j2) in flash_timer else (
                  GOLD if (i2,j2) in path_pairs else TEXT_C)
            vl  = F_SML.render(vs,True,col)
            screen.blit(vl,(cx+MAT_CELL//2-vl.get_width()//2,
                            cy+MAT_CELL//2-vl.get_height()//2))

    # legend
    ly = 62 + N*MAT_CELL + 10
    for col2,lbl in [(NODE_K,"k (index)"),(NODE_I,"i (rresht)"),(NODE_J,"j (kolonë)")]:
        pygame.draw.circle(screen,col2,(MAT_X+10,ly+8),8)
        s=F_SML.render(lbl,True,(160,180,220))
        screen.blit(s,(MAT_X+24,ly)); ly+=22

def advance_step():
    global step_idx, D_cur, k_cur, i_cur, j_cur, step_msg, phase
    if step_idx >= len(steps):
        phase = "done"; step_msg = "Floyd-Warshall përfundoi! Kliko 2 nyje për rrugën."; return
    s = steps[step_idx]
    k_cur,i_cur,j_cur = s[1],s[2],s[3]
    D_cur = [row[:] for row in s[5]]
    if s[0] == "update":
        nd = s[4]
        flash_timer[(i_cur,j_cur)] = 1.5
        step_msg = f"k={CITIES[k_cur]}: D[{CITIES[i_cur]}][{CITIES[j_cur]}] → {nd}"
    else:
        step_msg = f"k={CITIES[k_cur]}: D[{CITIES[i_cur]}][{CITIES[j_cur]}] = {D_cur[i_cur][j_cur]} (no change)"
    step_idx += 1

path_nodes = None

running = True
while running:
    dt = clock.tick(60)/1000.0

    # update flash timers
    for key in list(flash_timer.keys()):
        flash_timer[key] -= dt
        if flash_timer[key] <= 0:
            del flash_timer[key]

    for event in pygame.event.get():
        if event.type == pygame.QUIT or (
                event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE):
            running = False
        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_SPACE:
                advance_step(); auto_play=False
            elif event.key == pygame.K_a:
                auto_play = not auto_play
                step_msg = "Auto: " + ("ON (rapide)" if auto_play else "OFF")
            elif event.key == pygame.K_EQUALS or event.key == pygame.K_PLUS:
                speed = max(0.02, speed-0.02)
            elif event.key == pygame.K_MINUS:
                speed = min(0.5, speed+0.05)
            elif event.key == pygame.K_r:
                D_cur=[row[:] for row in ADJ]; k_cur=i_cur=j_cur=-1
                step_idx=0; phase="ready"; step_msg=""; auto_play=False
                flash_timer.clear(); path_nodes=None; sel_src=sel_dst=None
        elif event.type == pygame.MOUSEBUTTONDOWN and event.button==1:
            mx,my = event.pos
            if mx < GRAPH_W:
                for idx2,name in enumerate(CITIES):
                    pos2 = GRAPH_POS[name]
                    if math.hypot(mx-pos2[0],my-pos2[1])<24:
                        if sel_src is None: sel_src=idx2; step_msg=f"Burim: {name}. Kliko dest."
                        elif sel_dst is None and idx2!=sel_src:
                            sel_dst=idx2
                            p=get_path(sel_src,sel_dst,P_final)
                            path_nodes=p
                            d=D_final[sel_src][sel_dst]
                            step_msg=f"{CITIES[sel_src]}→{CITIES[sel_dst]}: {d if d<INF else '∞'}"
                        else:
                            sel_src=idx2; sel_dst=None; path_nodes=None
                            step_msg=f"Burim: {name}. Kliko dest."
                        break

    if auto_play and phase!="done":
        now = time.time()
        if now-last_auto > speed:
            last_auto=now; advance_step()

    # draw
    screen.fill(BG)
    pygame.draw.rect(screen,PANEL_BG,(GRAPH_W,0,W-GRAPH_W,H))
    pygame.draw.line(screen,(40,60,90),(GRAPH_W,0),(GRAPH_W,H),2)

    draw_graph(path_nodes)
    draw_matrix(D_cur, path_nodes)

    # left panel controls
    cx,cy = 10, H-160
    pygame.draw.line(screen,(40,60,90),(0,cy),(GRAPH_W,cy)); cy+=6
    for k2,v2 in [("SPACE","Hap tjetër"),("A","Auto-luaj"),
                  ("+/-","Shpejtësi"),("R","Rifillo"),("Klik 2 nyje","Shfaq rrugën")]:
        s2=F_SML.render(f"[{k2}] {v2}",True,(150,170,210))
        screen.blit(s2,(cx,cy)); cy+=17

    # progress
    prog = F_SML.render(f"Hapi: {step_idx}/{len(steps)}  Shpejtësi: {1/max(speed,0.01):.0f}x",True,(140,160,200))
    screen.blit(prog,(10,H-20))

    # step message
    if step_msg:
        sm2 = F_MED.render(step_msg[:50],True,GOLD)
        screen.blit(sm2,(10,10))

    # title
    tl=F_TITLE.render("Floyd-Warshall — Të gjitha çiftet",True,WHITE)
    screen.blit(tl,(GRAPH_W//2-tl.get_width()//2,H-25))

    esc=F_SML.render("ESC = dil",True,(80,90,110))
    screen.blit(esc,(W-90,H-20))

    pygame.display.flip()

pygame.quit()
sys.exit()


SystemExit: 

---

## Ushtrimi 6 — Floyd-Warshall: Gara e Rrugëve

### Teoria: Rindërtimi i rrugës

Pas ekzekutimit të Floyd-Warshall, përveç **distancës minimale** D[i][j], ruajmë edhe **matricën e paraardhësve** P[i][j] — kjo na lejon të rindërtojmë rrugën aktuale:

```python
def get_path(P, src, dst):
    path = [src]
    cur  = src
    while cur != dst:
        cur = P[cur][dst]  # merr nyjen tjetër
        path.append(cur)
    return path
```

### Loja — 8 Raunde

Ju paraqitet një çift qytetesh **(src → dst)**. Duhet të **vizatoni rrugën optimale** duke klikuar nyjet në radhë. Pasi mbaroni, shtypni `SPACE` për konfirmim.

**Pikëzimi:**
-  Rrugë optimale = **+15 pikë**
-  Rrugë e vlefshme (por jo optimale) = pikë të reduktuara
-  Rrugë e pavlefshme = 0 pikë

**Kontrollet:**
- Klikoni nyjet për të ndërtuar rrugën
- `SPACE` — konfirmo  `BACK` — hiq nyjen e fundit
- `N` — raundi tjetër  `R` — rifillo rrugën  | `ESC` — dil

In [6]:
import pygame, math, sys, random, time

pygame.init()
W, H = 1280, 720
screen = pygame.display.set_mode((0, 0), pygame.FULLSCREEN)
W, H = screen.get_size()
pygame.display.set_caption("Ushtrimi 6 — Floyd-Warshall: Gara e Rrugëve")
clock = pygame.time.Clock()

BG        = (8,  14,  28)
PANEL_BG  = (14, 22,  42)
NODE_C    = (50, 100, 190)
NODE_SEL  = (255, 160,  30)
NODE_OPT  = (50,  200,  80)
EDGE_C    = (55,  70, 110)
PATH_PLR  = (255, 220,  50)   # player path
PATH_OPT  = (80,  220, 100)   # optimal
TEXT_C    = (210, 225, 255)
GOLD      = (255, 215,   0)
WHITE     = (255, 255, 255)
GREEN     = (60,  210, 100)
RED       = (220,  60,  60)
BLUE      = (60,  130, 220)

F_TITLE = pygame.font.SysFont("consolas", 24, bold=True)
F_BIG   = pygame.font.SysFont("consolas", 20, bold=True)
F_MED   = pygame.font.SysFont("consolas", 16)
F_SML   = pygame.font.SysFont("consolas", 13)

INF = float('inf')
GRAPH_W = 820

CITIES = ["Alfa","Beta","Gama","Delta","Eta","Zeta","Iota","Kapa"]
N = len(CITIES)

# fixed positions on the 820×720 canvas
POSITIONS = [
    (140, 180), (340, 100), (560, 180), (740, 320),
    (140, 520), (340, 620), (560, 540), (700, 180),
]

def make_graph():
    ADJ = [[INF]*N for _ in range(N)]
    for i in range(N): ADJ[i][i] = 0
    fixed_edges = [
        (0,1,4),(1,2,3),(2,3,5),(3,7,6),(0,4,7),
        (1,4,8),(1,5,10),(2,6,4),(3,6,7),(4,5,3),
        (5,6,5),(6,7,4),(0,2,9),(4,6,6),(2,7,8),
    ]
    for u,v,w in fixed_edges:
        ADJ[u][v]=w; ADJ[v][u]=w
    return ADJ

def floyd_warshall(ADJ):
    D = [row[:] for row in ADJ]
    P = [[j if ADJ[i][j]<INF and i!=j else None for j in range(N)] for i in range(N)]
    for k in range(N):
        for i in range(N):
            for j in range(N):
                if D[i][k]<INF and D[k][j]<INF and D[i][k]+D[k][j]<D[i][j]:
                    D[i][j]=D[i][k]+D[k][j]; P[i][j]=P[i][k]
    return D, P

def get_path(P, src, dst):
    if P is None: return []
    path=[src]; cur=src
    visited=set([src])
    while cur!=dst:
        nxt=P[cur][dst]
        if nxt is None or nxt in visited: return []
        cur=nxt; path.append(cur); visited.add(cur)
    return path

ADJ = make_graph()
D, P = floyd_warshall(ADJ)

# rounds: pairs of (src, dst) indices
ALL_PAIRS = [(i,j) for i in range(N) for j in range(i+2, N) if D[i][j] < INF]
random.shuffle(ALL_PAIRS)
ROUNDS = ALL_PAIRS[:8]
TOTAL_ROUNDS = len(ROUNDS)

# ── state ─────────────────────────────────────────────────────────────────────
round_idx    = 0
score        = 0
player_path  = []    # indices
sel_node     = None
phase        = "playing"   # playing | result | gameover
show_optimal = False
step_msg     = ""
round_msg    = ""
time_limit   = 30.0
time_left    = time_limit
game_over    = False

def current_src_dst():
    return ROUNDS[round_idx]

def path_cost(path_list):
    if len(path_list) < 2: return INF
    total = 0
    for i in range(len(path_list)-1):
        c = ADJ[path_list[i]][path_list[i+1]]
        if c == INF: return INF
        total += c
    return total

def draw_graph(player_p, opt_p, src, dst):
    edges_drawn = set()
    # edges
    for u in range(N):
        for v in range(u+1, N):
            if ADJ[u][v] < INF:
                p1, p2 = POSITIONS[u], POSITIONS[v]
                in_plr = (u in player_p and v in player_p and
                          abs(player_p.index(u)-player_p.index(v))==1) if len(player_p)>1 else False
                in_opt = (u in opt_p and v in opt_p and
                          abs(opt_p.index(u)-opt_p.index(v))==1) if len(opt_p)>1 else False
                col = PATH_OPT if in_opt else (PATH_PLR if in_plr else EDGE_C)
                lw  = 5 if (in_opt or in_plr) else 2
                pygame.draw.line(screen, col, p1, p2, lw)
                mx,my=(p1[0]+p2[0])//2,(p1[1]+p2[1])//2
                wl=F_SML.render(str(ADJ[u][v]),True,(140,160,200))
                screen.blit(wl,(mx+3,my-8))

    # nodes
    for idx, name in enumerate(CITIES):
        pos = POSITIONS[idx]
        if idx == src: col = (255,80,50)
        elif idx == dst: col = (50,200,80)
        elif idx in player_p: col = (200,160,40)
        else: col = NODE_C
        r = 24
        pygame.draw.circle(screen, col, pos, r)
        if idx == sel_node:
            pygame.draw.circle(screen, WHITE, pos, r+3, 3)
        pygame.draw.circle(screen, WHITE, pos, r, 2)
        lbl = F_SML.render(name[:4], True, WHITE)
        screen.blit(lbl,(pos[0]-lbl.get_width()//2, pos[1]-lbl.get_height()//2))

    # player path cost
    if len(player_p) > 1:
        pc = path_cost(player_p)
        ct = F_SML.render(f"Kosto jote: {pc}", True, PATH_PLR)
        screen.blit(ct,(10,10))

def next_round():
    global round_idx, player_path, sel_node, phase, show_optimal, time_left, step_msg
    round_idx += 1
    if round_idx >= TOTAL_ROUNDS:
        phase = "gameover"
    else:
        player_path=[]; sel_node=None; phase="playing"
        show_optimal=False; time_left=time_limit
        src,dst = current_src_dst()
        step_msg = f"Raundi {round_idx+1}: gjej rrugën {CITIES[src]} → {CITIES[dst]}"

src_c, dst_c = current_src_dst()
step_msg = f"Raundi 1: gjej rrugën {CITIES[src_c]} → {CITIES[dst_c]}"

running = True
while running:
    dt = clock.tick(60)/1000.0

    if phase == "playing" and not game_over:
        time_left -= dt
        if time_left <= 0:
            time_left = 0
            phase = "result"
            show_optimal = True
            src_i, dst_i = current_src_dst()
            opt = get_path(P, src_i, dst_i)
            step_msg = f"Koha mbaroi! Optimale: {' → '.join(CITIES[o] for o in opt)}"

    for event in pygame.event.get():
        if event.type == pygame.QUIT or (
                event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE):
            running = False

        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_SPACE and phase == "playing":
                # confirm path
                src_i, dst_i = current_src_dst()
                if player_path and player_path[0]==src_i and player_path[-1]==dst_i:
                    pc = path_cost(player_path)
                    opt_c = D[src_i][dst_i]
                    phase = "result"; show_optimal=True
                    if pc == opt_c:
                        score += 15
                        step_msg = f"🎉 Perfekt! +15 pikë. Kosto: {pc}"
                    elif pc < INF:
                        pts = max(0, 8 - int((pc-opt_c)//2))
                        score += pts
                        step_msg = f"Rrugë e vlefshme (+{pts}). Optimale: {opt_c}, Jotja: {pc}"
                    else:
                        step_msg = "Rrugë e pavlefshme! Shih optimalen."
                else:
                    step_msg = f"Fillo nga {CITIES[src_i]} dhe mbyllo në {CITIES[dst_i]}!"

            elif event.key == pygame.K_r and phase in ("result","playing"):
                player_path=[]; sel_node=None
                if phase=="result": phase="playing"; show_optimal=False; time_left=20.0
                step_msg="Provoje sërish!"

            elif event.key == pygame.K_n and phase=="result":
                next_round()

            elif event.key == pygame.K_BACKSPACE and phase=="playing":
                if player_path: player_path.pop()
                sel_node = player_path[-1] if player_path else None

        elif event.type == pygame.MOUSEBUTTONDOWN and event.button==1:
            mx,my = event.pos
            if mx < GRAPH_W and phase=="playing":
                src_i, dst_i = current_src_dst()
                for idx,pos in enumerate(POSITIONS):
                    if math.hypot(mx-pos[0],my-pos[1])<26:
                        if not player_path:
                            if idx==src_i:
                                player_path=[idx]; sel_node=idx
                                step_msg=f"Fillova nga {CITIES[idx]}. Kliko nyjen tjetër."
                            else:
                                step_msg=f"Fillo nga {CITIES[src_i]}!"
                        else:
                            last=player_path[-1]
                            if idx==last: break
                            if ADJ[last][idx]<INF:
                                if idx not in player_path:
                                    player_path.append(idx); sel_node=idx
                                    step_msg=f"Shtova {CITIES[idx]}. Shtyp SPACE nëse ke mbaruar."
                                else:
                                    step_msg="Nyja tashmë në rrugë!"
                            else:
                                step_msg=f"Nuk ka lidhje direkte {CITIES[last]}→{CITIES[idx]}!"
                        break

    # ── draw ──────────────────────────────────────────────────────────────────
    screen.fill(BG)
    pygame.draw.rect(screen, PANEL_BG, (GRAPH_W,0,W-GRAPH_W,H))
    pygame.draw.line(screen,(40,60,90),(GRAPH_W,0),(GRAPH_W,H),2)

    if phase == "gameover":
        screen.fill((8,12,25))
        tl = F_TITLE.render("GARA MBAROI!", True, GOLD)
        screen.blit(tl,(W//2-tl.get_width()//2, H//2-100))
        sc = F_BIG.render(f"Rezultati: {score} / {TOTAL_ROUNDS*15}", True, WHITE)
        screen.blit(sc,(W//2-sc.get_width()//2,H//2-40))
        grade = "Shkëlqyer!" if score >= TOTAL_ROUNDS*12 else (
                "Shumë mirë!" if score >= TOTAL_ROUNDS*8 else "Provo sërish!")
        gl = F_BIG.render(grade, True, GREEN if score>=TOTAL_ROUNDS*8 else RED)
        screen.blit(gl,(W//2-gl.get_width()//2,H//2+10))
        rb = pygame.Rect(W//2-110,H//2+80,220,50)
        pygame.draw.rect(screen,BLUE,rb,border_radius=10)
        rt = F_MED.render("Luaj Sërish (R)",True,WHITE)
        screen.blit(rt,(rb.centerx-rt.get_width()//2,rb.centery-rt.get_height()//2))

        for event2 in []: pass  # handled above
        if any(event2.type==pygame.KEYDOWN and event2.key==pygame.K_r
               for event2 in pygame.event.get()):
            # restart
            round_idx=0; score=0; player_path=[]; sel_node=None
            phase="playing"; show_optimal=False; time_left=time_limit
            src_c2,dst_c2=current_src_dst()
            step_msg=f"Raundi 1: {CITIES[src_c2]} → {CITIES[dst_c2]}"

        pygame.display.flip(); continue

    src_i, dst_i = current_src_dst()
    opt_path = get_path(P, src_i, dst_i) if show_optimal else []
    draw_graph(player_path, opt_path, src_i, dst_i)

    # right panel
    px,py = GRAPH_W+12,12
    title = F_TITLE.render("Gara Floyd-Warshall", True, GOLD)
    screen.blit(title,(px,py)); py+=38

    sc_s = F_BIG.render(f"Pikët: {score}", True, GOLD)
    screen.blit(sc_s,(px,py))
    rp = F_MED.render(f"Raundi {round_idx+1}/{TOTAL_ROUNDS}",True,(160,180,220))
    screen.blit(rp,(W-rp.get_width()-10,py)); py+=30

    # timer bar
    bar_w = W-GRAPH_W-24
    pygame.draw.rect(screen,(30,40,60),(px,py,bar_w,12),border_radius=6)
    frac = max(0,time_left/time_limit)
    col_t = GREEN if frac>0.5 else (GOLD if frac>0.25 else RED)
    pygame.draw.rect(screen,col_t,(px,py,int(bar_w*frac),12),border_radius=6)
    py+=20

    pygame.draw.line(screen,(40,60,90),(px,py),(W-10,py)); py+=8

    # objective
    obj1 = F_MED.render(f"Gjej rrugën:", True, WHITE)
    screen.blit(obj1,(px,py)); py+=22
    src_lbl = F_BIG.render(f"{CITIES[src_i]}", True, (255,80,50))
    screen.blit(src_lbl,(px,py)); py+=26
    arr = F_MED.render("  ↓", True, (160,180,220))
    screen.blit(arr,(px,py)); py+=22
    dst_lbl = F_BIG.render(f"{CITIES[dst_i]}", True, (50,200,80))
    screen.blit(dst_lbl,(px,py)); py+=30

    # optimal cost (hint)
    if show_optimal:
        oc = F_MED.render(f"Optimale: {D[src_i][dst_i]}", True, PATH_OPT)
        screen.blit(oc,(px,py)); py+=22
        op_str = " → ".join(CITIES[x] for x in opt_path)
        words=op_str.split(); line=""; lines=[]
        for w in words:
            if len(line)+len(w)+1>20: lines.append(line); line=w
            else: line=(line+" "+w).strip()
        if line: lines.append(line)
        for ln in lines:
            sl=F_SML.render(ln,True,PATH_OPT); screen.blit(sl,(px,py)); py+=16
        py+=6

    pygame.draw.line(screen,(40,60,90),(px,py),(W-10,py)); py+=8

    controls2 = [
        ("Klik","Shto nyje në rrugë"),
        ("SPACE","Konfirmo rrugën"),
        ("BACK","Hiq nyjen e fundit"),
        ("N","Raundi tjetër"),
        ("R","Rifillo rrugën"),
    ]
    for k2,v2 in controls2:
        s2=F_SML.render(f"[{k2}] {v2}",True,(150,170,210))
        screen.blit(s2,(px,py)); py+=18

    pygame.draw.line(screen,(40,60,90),(px,py),(W-10,py)); py+=8

    # message
    if step_msg:
        words2=step_msg.split(); line2=""; lines2=[]
        for w2 in words2:
            if len(line2)+len(w2)+1>22: lines2.append(line2); line2=w2
            else: line2=(line2+" "+w2).strip()
        if line2: lines2.append(line2)
        for ln in lines2:
            sl2=F_SML.render(ln,True,(220,240,180)); screen.blit(sl2,(px,py)); py+=16

    esc=F_SML.render("ESC = dil",True,(80,90,110))
    screen.blit(esc,(GRAPH_W+10,H-20))

    pygame.display.flip()

pygame.quit()
sys.exit()


SystemExit: 

---

## Punë në Klasë

### Problemi 1 — Dijkstra me dorë

Jepet grafi i drejtëzuar me peshe:

```
A --4--> B --3--> D
|        |        |
6        2        1
|        v        v
+------> C --1--> E --2--> F
```

Brinjët: A→B(4), A→C(6), B→C(2), B→D(3), C→E(1), D→E(1), D→F(7), E→F(2)

**Detyrë:** Ekzekutoni Dijkstrën nga kulmi **A**. Plotësoni tabelën:

| Iterimi | Kulmi i zgjedhur | dist[A] | dist[B] | dist[C] | dist[D] | dist[E] | dist[F] |
|---------|:-----------------:|:-------:|:-------:|:-------:|:-------:|:-------:|:-------:|
| Fillimi | — | 0 | ∞ | ∞ | ∞ | ∞ | ∞ |
| 1 | ? | | | | | | |
| 2 | ? | | | | | | |
| 3 | ? | | | | | | |
| 4 | ? | | | | | | |
| 5 | ? | | | | | | |
| 6 | ? | | | | | | |

---

### Problemi 2 — Bellman-Ford dhe Ciklet Negative

Jepet grafi i drejtëzuar:

```
A --1--> B --(-4)--> C
^                    |
|        3           2
+--------D <--(-1)---+
```

Brinjët: A→B(1), B→C(-4), C→D(2), D→A(3), C→D(-1)

**Detyrë:**
1. Ekzekutoni Bellman-Ford nga A (bëni 4 iterime).
2. A ka cikël negativ? Identifikojeni atë.
3. Çfarë ndodh me dist[] nëse vazhdojmë iterime?

---

### Problemi 3 — Floyd-Warshall me dorë

Jepet matrica e distancave fillestare (4 kulme):

```
D⁰ =  [ 0   3   ∞   7 ]
       [ 8   0   2   ∞ ]
       [ 5   ∞   0   1 ]
       [ 2   ∞   ∞   0 ]
```

**Detyrë:** Llogaritni D¹ (k=0), D² (k=1), D³ (k=2), D⁴ (k=3).

*Kujtoni: D[i][j] = min(D[i][j], D[i][k] + D[k][j])*

---

### Problemi 4 — Krahasimi i algoritmeve

Plotësoni tabelën e krahasimit:

| Kriteret | Dijkstra | Bellman-Ford | Floyd-Warshall |
|----------|----------|-------------|---------------|
| Peshe negative | | | |
| Zbulim cikli negativ | | | |
| Burim(e) | | | |
| Kompleksiteti kohor | | | |
| Kompleksiteti hapësinor | | | |
| Zbatim i preferuar | | | |

---